# Price metrics and incremental strategy comparison

This notebook demonstrates how momentum, trend, and volatility become explicit inputs to a declarative strategy. A metric is an observable; the strategy is the rule set that converts observables into target portfolio weights.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cobasket.price_metrics import (
    PriceMetricConfig,
    build_price_metrics,
    compare_incremental_metric_strategies,
)
from cobasket.strategy_rules import MetricCondition, StrategyRule, StrategyRules

## Synthetic price history

The example is synthetic so it runs without network access. Replace `prices` with an aligned adjusted-price table for a real basket.

In [ ]:
index = pd.date_range('2020-01-02', periods=900, freq='B')
rng = np.random.default_rng(42)
market = np.cumsum(rng.normal(0.0003, 0.008, len(index)))
prices = pd.DataFrame({
    'AAA': 100 * np.exp(market + np.cumsum(rng.normal(0, 0.003, len(index)))),
    'BBB': 100 * np.exp(0.8 * market + np.cumsum(rng.normal(0, 0.004, len(index)))),
}, index=index)
prices.plot(title='Synthetic adjusted prices');

## Build trailing metrics

All windows are backward-looking. Early rows remain missing until enough history is available. `momentum` and `trend` are bounded scores in [-1, 1]. `volatility` is annualised return scatter, not a directional forecast.

In [ ]:
price_metrics = build_price_metrics(
    prices,
    config=PriceMetricConfig(
        momentum_window=60,
        trend_window=100,
        volatility_window=20,
        volatility_baseline_window=252,
    ),
)
pd.concat({name: table['AAA'] for name, table in price_metrics.items()}, axis=1).tail()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
price_metrics['momentum'].plot(ax=axes[0], title='Momentum score')
price_metrics['trend'].plot(ax=axes[1], title='Trend score')
price_metrics['volatility'].plot(ax=axes[2], title='Annualised volatility')
plt.tight_layout()

## Define strategies before comparing them

For illustration, `probability` is a synthetic table. In a real run it would come from Cobasket's walk-forward cointegration calibration. The strategies differ by one added condition at a time.

In [ ]:
evaluation_dates = prices.index[300::20]
probability = pd.DataFrame(0.65, index=evaluation_dates, columns=prices.columns)
stable = pd.DataFrame(1.0, index=evaluation_dates, columns=prices.columns)

probability_only = StrategyRules(
    name='probability only',
    rules=(
        StrategyRule('sell', (MetricCondition('probability', '<=', 0.30),), 0.0),
        StrategyRule('buy', (MetricCondition('probability', '>=', 0.60),), 0.20),
    ),
)

with_momentum = StrategyRules(
    name='probability + momentum',
    rules=(
        StrategyRule('sell', (MetricCondition('probability', '<=', 0.30),), 0.0),
        StrategyRule('buy', (
            MetricCondition('probability', '>=', 0.60),
            MetricCondition('momentum', '>', 0.0),
        ), 0.20),
    ),
)

with_risk_filter = StrategyRules(
    name='probability + momentum + volatility filter',
    rules=(
        StrategyRule('sell', (MetricCondition('probability', '<=', 0.30),), 0.0),
        StrategyRule('buy', (
            MetricCondition('probability', '>=', 0.60),
            MetricCondition('stable', '==', True),
            MetricCondition('momentum', '>', 0.0),
            MetricCondition('high_volatility', '==', False),
        ), 0.20),
    ),
)

In [ ]:
summary, results = compare_incremental_metric_strategies(
    prices,
    {'probability': probability, 'stable': stable},
    price_metrics,
    (probability_only, with_momentum, with_risk_filter),
    initial_cash=10_000.0,
)
summary

In [ ]:
for name, result in results.items():
    result.backtest.equity.plot(label=name)
plt.ylabel('Portfolio value')
plt.legend()
plt.title('Identical data and execution assumptions; different rules');

## Interpretation

A more restrictive strategy may trade less, reduce drawdown, or miss profitable periods. The relevant question is not whether a metric looks plausible, but whether adding its pre-declared rule improves held-out performance after transaction costs. Avoid repeatedly changing thresholds after inspecting the final test period.